In [2]:
import sys
from pathlib import Path
import pandas as pd

# Find the repository root.
PROJECT_ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "requirements.txt").is_file()
        and (path / "Datasets").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("ClearFlow project root not found.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.schema import SDN_TO_CICIDS

DATASET_DIRS = {
    "CICIDS-2017": PROJECT_ROOT / "Datasets" / "CICIDS2017",
    "SDN TCP-SYN": PROJECT_ROOT / "Datasets" / "SDN_TCP_SYN",
}

print("Project root:", PROJECT_ROOT)
print("Feature mappings loaded:", len(SDN_TO_CICIDS))
print("CICIDS files:", len(list(DATASET_DIRS["CICIDS-2017"].glob("*.csv"))))
print("SDN TCP-SYN files:", len(list(DATASET_DIRS["SDN TCP-SYN"].glob("*.csv"))))

Project root: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow
Feature mappings loaded: 50
CICIDS files: 8
SDN TCP-SYN files: 1


In [3]:
import pandas as pd

def normalize(name):
    return str(name).strip().casefold()


aliases = {
    normalize(source): normalize(target)
    for source, target in SDN_TO_CICIDS.items()
}

excluded = {
    "label", "flow id",
    "src ip", "dst ip", "source ip", "destination ip",
    "src port", "dst port", "source port", "destination port",
    "timestamp", "protocol",
    "fwd header length.1",
}

cicids_files = sorted(DATASET_DIRS["CICIDS-2017"].glob("*.csv"))
sdn_files = sorted(DATASET_DIRS["SDN TCP-SYN"].glob("*.csv"))

reference_columns = pd.read_csv(
    cicids_files[0], nrows=0, encoding="utf-8-sig"
).columns.tolist()

canonical_lookup = {
    normalize(column): column.strip()
    for column in reference_columns
    if normalize(column) not in excluded
}

FEATURE_COLUMNS = list(canonical_lookup.values())

assert len(FEATURE_COLUMNS) == 76, (
    f"Expected 76 features, found {len(FEATURE_COLUMNS)}"
)


def map_label(value, dataset):
    if pd.isna(value):
        return pd.NA

    label = str(value).strip().replace("\ufffd", "-").casefold()

    if label in {"", "<missing>"}:
        return pd.NA

    if dataset == "SDN TCP-SYN":
        if label == "normal":
            return "Benign"
        if label == "ddos":
            return "SYN Flood"

    label_mapping = {
        "benign": "Benign",
        "ddos": "DDoS",
        "portscan": "PortScan",
        "dos hulk": "DoS Hulk",
        "dos goldeneye": "DoS GoldenEye",
        "ftp-patator": "FTP-Patator",
        "ssh-patator": "SSH-Patator",
        "dos slowloris": "DoS slowloris",
        "dos slowhttptest": "DoS Slowhttptest",
        "bot": "Bot",
        "infiltration": "Infiltration",
        "heartbleed": "Heartbleed",
        "web attack - brute force": "Web Attack",
        "web attack - sql injection": "Web Attack",
        "web attack - xss": "Web Attack",
    }

    if label not in label_mapping:
        raise ValueError(f"Unknown label: {value!r} in {dataset}")

    return label_mapping[label]


def validate_file(path, dataset):
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
    label_columns = [
        column for column in header.columns
        if normalize(column) == "label"
    ]

    if len(label_columns) != 1:
        raise ValueError(f"Unexpected label columns in {path.name}")

    # Sample rows to test the actual column transformation.
    sample = pd.read_csv(
        path, nrows=1000, encoding="utf-8-sig", low_memory=False
    )

    rename = {}

    for column in sample.columns:
        source = normalize(column)

        if source in excluded:
            continue

        target = (
            aliases.get(source, source)
            if dataset == "SDN TCP-SYN"
            else source
        )

        if target not in canonical_lookup:
            raise ValueError(
                f"Unmapped feature {column!r} in {path.name}"
            )

        rename[column] = canonical_lookup[target]

    if len(set(rename.values())) != len(rename):
        raise ValueError(f"Feature-name collision in {path.name}")

    X = sample[list(rename)].rename(columns=rename)

    if set(X.columns) != set(FEATURE_COLUMNS):
        missing = sorted(set(FEATURE_COLUMNS) - set(X.columns))
        extra = sorted(set(X.columns) - set(FEATURE_COLUMNS))
        raise ValueError(
            f"Schema mismatch in {path.name}; "
            f"missing={missing}, extra={extra}"
        )

    X = X[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    y = sample[label_columns[0]].map(
        lambda value: map_label(value, dataset)
    )

    return {
        "dataset": dataset,
        "file": path.name,
        "sample_rows": len(sample),
        "feature_count": X.shape[1],
        "mapped_labels": y.nunique(dropna=True),
        "missing_labels_in_sample": int(y.isna().sum()),
    }


validation_results = []

for path in cicids_files:
    validation_results.append(validate_file(path, "CICIDS-2017"))

for path in sdn_files:
    validation_results.append(validate_file(path, "SDN TCP-SYN"))

display(pd.DataFrame(validation_results))

,dataset,file,sample_rows,feature_count,mapped_labels,missing_labels_in_sample
0,CICIDS-2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,1000,76,1,0
1,CICIDS-2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,1000,76,1,0
2,CICIDS-2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,1000,76,1,0
3,CICIDS-2017,Monday-WorkingHours.pcap_ISCX.csv,1000,76,1,0
4,CICIDS-2017,Thursday-WorkingHours-Afternoon-Infilteration....,1000,76,1,0
5,CICIDS-2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,1000,76,1,0
6,CICIDS-2017,Tuesday-WorkingHours.pcap_ISCX.csv,1000,76,1,0
7,CICIDS-2017,Wednesday-workingHours.pcap_ISCX.csv,1000,76,1,0
8,SDN TCP-SYN,TCP-SYNC DATASET.csv,1000,76,1,0


In [4]:
from collections import Counter
import pandas as pd

label_audit = []

for dataset, directory in DATASET_DIRS.items():
    counts = Counter()

    for path in sorted(directory.glob("*.csv")):
        for chunk in pd.read_csv(
            path,
            usecols=lambda c: c.strip().casefold() == "label",
            chunksize=250_000,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
        ):
            label_column = chunk.columns[0]

            for raw_label, count in chunk[label_column].value_counts(
                dropna=False
            ).items():
                counts[str(raw_label)] += int(count)

    print(f"\n{dataset}: all raw labels")
    for label, count in sorted(counts.items()):
        mapped = map_label(label, dataset)
        print(f"{label!r:40} -> {str(mapped):25} {count:>10,}")

        label_audit.append({
            "dataset": dataset,
            "raw_label": label,
            "mapped_label": mapped,
            "count": count,
        })

label_mapping_audit = pd.DataFrame(label_audit)

output_path = (
    PROJECT_ROOT / "results" / "metrics"
    / "full_label_mapping_audit.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
label_mapping_audit.to_csv(output_path, index=False)

print(f"\nSaved full label audit to: {output_path}")


CICIDS-2017: all raw labels
'BENIGN'                                 -> Benign                     2,273,097
'Bot'                                    -> Bot                            1,966
'DDoS'                                   -> DDoS                         128,027
'DoS GoldenEye'                          -> DoS GoldenEye                 10,293
'DoS Hulk'                               -> DoS Hulk                     231,073
'DoS Slowhttptest'                       -> DoS Slowhttptest               5,499
'DoS slowloris'                          -> DoS slowloris                  5,796
'FTP-Patator'                            -> FTP-Patator                    7,938
'Heartbleed'                             -> Heartbleed                        11
'Infiltration'                           -> Infiltration                      36
'PortScan'                               -> PortScan                     158,930
'SSH-Patator'                            -> SSH-Patator                    5,897

In [6]:
def harmonize_chunk(frame, dataset):
    """Align feature columns and standardize labels for one CSV chunk."""

    label_columns = [
        col for col in frame.columns
        if normalize(col) == "label"
    ]

    if len(label_columns) != 1:
        raise ValueError(
            f"Expected exactly one label column in {dataset}; "
            f"found {label_columns}"
        )

    label_column = label_columns[0]

    y = frame[label_column].map(
        lambda value: map_label(value, dataset)
    )

    rename = {}

    for column in frame.columns:
        source = normalize(column)

        # Exclude identifiers, metadata, labels and redundant columns.
        if source in excluded:
            continue

        target = (
            aliases.get(source, source)
            if dataset == "SDN TCP-SYN"
            else source
        )

        if target not in canonical_lookup:
            raise ValueError(
                f"Unmapped feature {column!r} in {dataset}"
            )

        rename[column] = canonical_lookup[target]

    if len(set(rename.values())) != len(rename):
        raise ValueError(
            f"Feature-name collision detected in {dataset}"
        )

    X = frame[list(rename)].rename(columns=rename)
    X = X[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")

    return X, y


print("harmonize_chunk() defined successfully.")

harmonize_chunk() defined successfully.


In [9]:
# Sensitivity check: retain Destination Port as a model feature.
# Source ports, IPs, timestamps, flow IDs, protocol, and the
# redundant Fwd Header Length.1 remain excluded.

excluded.discard("dst port")
excluded.discard("destination port")

reference_columns = pd.read_csv(
    next(DATASET_DIRS["CICIDS-2017"].glob("*.csv")),
    nrows=0,
    encoding="utf-8-sig",
).columns.tolist()

canonical_lookup = {
    normalize(column): column.strip()
    for column in reference_columns
    if normalize(column) not in excluded
}

FEATURE_COLUMNS = list(canonical_lookup.values())

print("Candidate model features:", len(FEATURE_COLUMNS))
print(
    "Destination Port retained:",
    any(normalize(c) == "destination port" for c in FEATURE_COLUMNS),
)

Candidate model features: 77
Destination Port retained: True


In [10]:
import os
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


CHUNK_SIZE = 100_000
EXCLUDED_LABELS = {"Heartbleed", "Infiltration"}

# Create a temporary SQLite database outside the repository.
fd, db_path = tempfile.mkstemp(
    prefix="clearflow_model_audit_",
    suffix=".sqlite",
)
os.close(fd)

con = sqlite3.connect(db_path)

try:
    con.execute("""
        CREATE TABLE signatures (
            dataset TEXT NOT NULL,
            feature_hash TEXT NOT NULL,
            label TEXT NOT NULL,
            occurrences INTEGER NOT NULL,
            PRIMARY KEY (dataset, feature_hash, label)
        ) WITHOUT ROWID
    """)

    stage_records = []

    for dataset, directory in DATASET_DIRS.items():
        for path in sorted(directory.glob("*.csv")):
            print(f"Auditing modeling rows: {path.name}")

            file_rows = 0
            missing_labels = 0
            excluded_class_rows = 0
            invalid_feature_rows = 0
            eligible_rows = 0

            for chunk in pd.read_csv(
                path,
                chunksize=CHUNK_SIZE,
                low_memory=False,
                encoding="utf-8-sig",
            ):
                # Standardize feature names and labels.
                X, y = harmonize_chunk(chunk, dataset)
                file_rows += len(chunk)

                missing_label = y.isna()
                excluded_label = (
                    y.isin(EXCLUDED_LABELS).fillna(False)
                )

                missing_labels += int(missing_label.sum())
                excluded_class_rows += int(excluded_label.sum())

                # Treat infinities as invalid feature values.
                X = X.replace([np.inf, -np.inf], np.nan)

                # IMPORTANT: enforce identical numeric dtypes
                # across chunks before calculating row hashes.
                X = X.astype("float64")

                invalid_features = X.isna().any(axis=1)

                eligible_before_quality = (
                    ~missing_label & ~excluded_label
                )

                invalid_feature_rows += int(
                    (
                        eligible_before_quality
                        & invalid_features
                    ).sum()
                )

                # Only retain rows with valid labels and features.
                eligible = (
                    eligible_before_quality
                    & ~invalid_features
                )

                if not eligible.any():
                    continue

                X_valid = X.loc[eligible]
                y_valid = y.loc[eligible].astype(str)

                # Hash feature vectors without identifiers or metadata.
                feature_hashes = pd.util.hash_pandas_object(
                    X_valid,
                    index=False,
                ).astype("uint64")

                pairs = pd.DataFrame({
                    "feature_hash": feature_hashes.astype(str).to_numpy(),
                    "label": y_valid.to_numpy(),
                })

                counts = (
                    pairs.groupby(
                        ["feature_hash", "label"],
                        sort=False,
                    )
                    .size()
                    .reset_index(name="occurrences")
                )

                con.executemany(
                    """
                    INSERT INTO signatures
                        (dataset, feature_hash, label, occurrences)
                    VALUES (?, ?, ?, ?)
                    ON CONFLICT(dataset, feature_hash, label)
                    DO UPDATE SET
                        occurrences =
                        occurrences + excluded.occurrences
                    """,
                    [
                        (
                            dataset,
                            row.feature_hash,
                            row.label,
                            int(row.occurrences),
                        )
                        for row in counts.itertuples(index=False)
                    ],
                )

                eligible_rows += int(eligible.sum())

            con.commit()

            stage_records.append({
                "dataset": dataset,
                "file": path.name,
                "raw_rows": file_rows,
                "missing_label_rows": missing_labels,
                "excluded_class_rows": excluded_class_rows,
                "invalid_feature_rows": invalid_feature_rows,
                "eligible_rows_before_dedup": eligible_rows,
            })

    # ---------------------------------------------------------
    # 1. Rows surviving label and feature validation
    # ---------------------------------------------------------
    stage_summary = pd.DataFrame(stage_records)

    print("\nROWS AFTER LABEL AND FEATURE VALIDATION")
    display(stage_summary)

    # ---------------------------------------------------------
    # 2. Repeated feature-label patterns across the merged data
    # ---------------------------------------------------------
    print("\nGLOBAL DUPLICATE SUMMARY")

    duplicate_summary = pd.read_sql_query("""
        WITH patterns AS (
            SELECT
                feature_hash,
                label,
                SUM(occurrences) AS occurrences
            FROM signatures
            GROUP BY feature_hash, label
        )
        SELECT
            SUM(occurrences) AS eligible_rows,
            COUNT(*) AS unique_feature_label_patterns,
            SUM(occurrences) - COUNT(*) AS duplicate_rows
        FROM patterns
    """, con)

    display(duplicate_summary)

    # ---------------------------------------------------------
    # 3. Feature vectors associated with multiple labels
    # ---------------------------------------------------------
    print("\nCONFLICTING LABEL SUMMARY")

    conflict_summary = pd.read_sql_query("""
        WITH patterns AS (
            SELECT
                feature_hash,
                label,
                SUM(occurrences) AS occurrences
            FROM signatures
            GROUP BY feature_hash, label
        ),
        conflicts AS (
            SELECT
                feature_hash,
                SUM(occurrences) AS affected_rows
            FROM patterns
            GROUP BY feature_hash
            HAVING COUNT(DISTINCT label) > 1
        )
        SELECT
            COUNT(*) AS conflicting_feature_patterns,
            COALESCE(SUM(affected_rows), 0) AS affected_rows
        FROM conflicts
    """, con)

    display(conflict_summary)

    # ---------------------------------------------------------
    # 4. Identify the labels involved in each conflict
    # ---------------------------------------------------------
    print("\nCONFLICTING LABEL PAIRS")

    conflict_pairs = pd.read_sql_query("""
        WITH patterns AS (
            SELECT
                feature_hash,
                label,
                SUM(occurrences) AS occurrences
            FROM signatures
            GROUP BY feature_hash, label
        )
        SELECT
            a.label AS label_a,
            b.label AS label_b,
            COUNT(*) AS conflicting_patterns,
            SUM(a.occurrences) AS rows_label_a,
            SUM(b.occurrences) AS rows_label_b
        FROM patterns a
        JOIN patterns b
          ON a.feature_hash = b.feature_hash
         AND a.label < b.label
        GROUP BY a.label, b.label
        ORDER BY conflicting_patterns DESC
    """, con)

    display(conflict_pairs)

finally:
    con.close()
    Path(db_path).unlink(missing_ok=True)

Auditing modeling rows: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Auditing modeling rows: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Auditing modeling rows: Friday-WorkingHours-Morning.pcap_ISCX.csv
Auditing modeling rows: Monday-WorkingHours.pcap_ISCX.csv
Auditing modeling rows: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Auditing modeling rows: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Auditing modeling rows: Tuesday-WorkingHours.pcap_ISCX.csv
Auditing modeling rows: Wednesday-workingHours.pcap_ISCX.csv
Auditing modeling rows: TCP-SYNC DATASET.csv

ROWS AFTER LABEL AND FEATURE VALIDATION


,dataset,file,raw_rows,missing_label_rows,excluded_class_rows,invalid_feature_rows,eligible_rows_before_dedup
0,CICIDS-2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,0,34,225711
1,CICIDS-2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,0,371,286096
2,CICIDS-2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,0,122,190911
3,CICIDS-2017,Monday-WorkingHours.pcap_ISCX.csv,529918,0,0,437,529481
4,CICIDS-2017,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,36,207,288359
5,CICIDS-2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,0,0,135,170231
6,CICIDS-2017,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,0,264,445645
7,CICIDS-2017,Wednesday-workingHours.pcap_ISCX.csv,692703,0,11,1297,691395
8,SDN TCP-SYN,TCP-SYNC DATASET.csv,6234,1,0,0,6233



GLOBAL DUPLICATE SUMMARY


,eligible_rows,unique_feature_label_patterns,duplicate_rows
0,2834062,2524138,309924



CONFLICTING LABEL SUMMARY


,conflicting_feature_patterns,affected_rows
0,697,6666



CONFLICTING LABEL PAIRS


,label_a,label_b,conflicting_patterns,rows_label_a,rows_label_b
0,Benign,PortScan,564,565,1053
1,Benign,DoS Hulk,129,161,4873
2,Benign,DDoS,3,3,6
3,Benign,DoS slowloris,1,4,1


In [1]:
import sys
from pathlib import Path
from IPython.display import display

# Locate the project root independently of Notebook 01.
PROJECT_ROOT = next(
    (
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "requirements.txt").is_file()
        and (p / "Datasets").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("ClearFlow project root not found.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import run_preprocessing

results = run_preprocessing(
    project_root=PROJECT_ROOT,
    chunk_size=100_000,
    sample_per_chunk_class=200,
    random_seed=42,
)

print("\nPREPROCESSING SUMMARY BY FILE")
display(results["summary_by_file"])

print("\nOVERALL PREPROCESSING SUMMARY")
display(results["summary_totals"])

print("\nZERO-VARIANCE FEATURES REMOVED")
display(
    results["feature_audit"].query("zero_variance")
)

print("\nCONFLICTING LABEL PAIRS")
display(results["conflict_pairs"])

print("\nClean dataset:", results["clean_path"])
print("BI exports:", results["export_dir"])

PASS 1/4: cleaning invalid rows and removing exact raw duplicates...
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv
  TCP-SYNC DATASET.csv
PASS 2/4: detecting conflicting labels for identical model inputs...
  Conflicting feature patterns: 697; rows to exclude: 1,394
PASS 3/4: excluding ambiguous feature vectors and profiling retained features...
  Zero-variance features removed: 7
   Bwd URG Flags, Fwd Avg Bytes/Bulk, Fwd Avg Packets/Bulk, Fwd Avg Bulk Rate, Bwd Avg Bytes/Bulk, Bwd Avg Packets/Bulk, Bwd Avg Bulk Rate
PASS 4/4: writing final clean dataset and BI exports...

Preprocessing complete.
Clean dataset: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV

,dataset,file,raw_rows,missing_label_rows,excluded_class_rows,invalid_feature_rows,eligible_before_dedup,exact_duplicate_rows_removed,rows_after_exact_dedup,conflicting_feature_rows_removed,rows_after_conflict_removal,final_rows
0,CICIDS-2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,0,34,225711,2629,223082,7,223075,223075
1,CICIDS-2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,0,371,286096,73842,212254,570,211684,211684
2,CICIDS-2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,0,122,190911,10814,180097,10,180087,180087
3,CICIDS-2017,Monday-WorkingHours.pcap_ISCX.csv,529918,0,0,437,529481,27017,502464,2,502462,502462
4,CICIDS-2017,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,36,207,288359,42539,245820,605,245215,245215
5,CICIDS-2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,0,0,135,170231,13652,156579,8,156571,156571
6,CICIDS-2017,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,0,264,445645,35662,409983,28,409955,409955
7,CICIDS-2017,Wednesday-workingHours.pcap_ISCX.csv,692703,0,11,1297,691395,100923,590472,164,590308,590308
8,SDN TCP-SYN,TCP-SYNC DATASET.csv,6234,1,0,0,6233,0,6233,0,6233,6233



OVERALL PREPROCESSING SUMMARY


,dataset,raw_rows,missing_label_rows,excluded_class_rows,invalid_feature_rows,eligible_before_dedup,exact_duplicate_rows_removed,rows_after_exact_dedup,conflicting_feature_rows_removed,rows_after_conflict_removal,final_rows
0,CICIDS-2017,2830743,0,47,2867,2827829,307078,2520751,1394,2519357,2519357
1,SDN TCP-SYN,6234,1,0,0,6233,0,6233,0,6233,6233
2,ALL,2836977,1,47,2867,2834062,307078,2526984,1394,2525590,2525590



ZERO-VARIANCE FEATURES REMOVED


,feature,min_after_cleaning,max_after_cleaning,zero_variance,decision,note
33,Bwd URG Flags,0.0,0.0,True,removed: zero variance,
55,Fwd Avg Bytes/Bulk,0.0,0.0,True,removed: zero variance,
56,Fwd Avg Packets/Bulk,0.0,0.0,True,removed: zero variance,
57,Fwd Avg Bulk Rate,0.0,0.0,True,removed: zero variance,
58,Bwd Avg Bytes/Bulk,0.0,0.0,True,removed: zero variance,
59,Bwd Avg Packets/Bulk,0.0,0.0,True,removed: zero variance,
60,Bwd Avg Bulk Rate,0.0,0.0,True,removed: zero variance,



CONFLICTING LABEL PAIRS


,label_a,label_b,conflicting_patterns,rows_label_a,rows_label_b
0,Benign,PortScan,564,564,564
1,Benign,DoS Hulk,129,129,129
2,Benign,DDoS,3,3,3
3,Benign,DoS slowloris,1,1,1



Clean dataset: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\data\processed\cleaned_flows.csv.gz
BI exports: C:\Users\arhat\OneDrive\Desktop\Amrita\sem5\DV\Project\ClearFlow\exports\phase02_clean
